# Layer 1: Data and Platform (class notes)

**What this layer is, in one line:** the stockroom. It collects and tidies everything the other four layers need, and it makes sure that later tests are fair. It predicts nothing.

These notes explain the ideas. You do not need to know how the code is written. You need to be able to explain, in plain words, what each piece is and why we did it.

## 1. The whole project, in plain words

After a flood, crop insurance (the scheme is called PMFBY) does not pay farmer by farmer. It pays per **unit**, usually a village or a group of villages. Our rule: **if more than 25 percent of a unit's rice (paddy) fields were flooded, the unit is paid.** That 25 percent is called the **loss line**, written **tau = 0.25**.

The problem: nobody can walk every field after a flood, and the roads are flooded too. So for every unit we need a quick answer, "above 25 percent" or "below", plus an honest idea of how sure we are.

Our answer is five layers, like five stages on a factory line:

| Layer | Job |
|---|---|
| **1 Data and platform** | collect and prepare everything, split it fairly (this one) |
| 2 Mapping | turn each satellite pixel into "probability this is flooded rice field" |
| 3 Estimation | combine the map with a few field visits to give each unit a number and a call |
| 4 Field planning | decide which villages to visit and which roads to drive |
| 5 Tool | the screen the officer uses |

Two rules apply to everything we do:

* **Every unit always gets a call.** We never say "cannot decide". If we are unsure, the probability attached to the call says so.
* **We say what is simulated.** The field checks and some other parts are simulated by computer, and we always label them.

## 2. Why radar? (what VV and VH are)

Normal satellite photos need sunlight and clear sky. During the monsoon there are clouds for weeks. So we use **radar** satellites (here **Sentinel-1**, from the European Space Agency). A radar satellite sends down its own signal and measures how much bounces back. It works day and night and sees through cloud.

**How water shows up.** Calm flood water acts like a mirror: the signal bounces away from the satellite instead of back to it. So very little comes back, and flooded ground looks **dark** in the radar picture. Land, crops and buildings scatter the signal in all directions, so more comes back and they look brighter.

**What VV and VH mean.** The radar signal has a direction of vibration, called its **polarisation**, either vertical (V) or horizontal (H). The satellite sends a signal and listens for the reply:

* **VV**: sent vertical, received vertical. This is the strongest reading and the best one for spotting open water.
* **VH**: sent vertical, received horizontal. This is weaker. It responds to vegetation and rough surfaces, so it helps tell crops from bare ground.

Think of them as two different "colours" of the same radar picture.

**What dB means.** Radar values are on a log scale called **decibels (dB)**. You only need to know: **lower (more negative) means darker**. Open water is typically below about **-15 dB**.

## 3. The practice material: labelled chips

To test or teach a flood detector you need pictures where a person has already marked "this is water". Those marks are **labels**. We use a public dataset called **Sen1Floods11**: small radar pictures called **chips** (128 by 128 pixels, each pixel 20 m across, so a chip covers about 2.5 km by 2.5 km), with water drawn in by hand. It covers **11 real flood events** around the world.

Not every chip is usable. After removing chips where too little ground could be judged, Layer 2 works with **121 chips**.

## 4. The most important idea: split by flood event, never by tile

When you build a method, you keep some data aside to test it, like keeping exam questions secret. The split is the piece most people get wrong.

* **Train**: the method learns from this (the textbook).
* **Validation**: we tune settings and check progress (practice tests).
* **Test**: scored **once**, at the very end (the final exam).

**The trap.** Chips from the same flood event look alike: same weather, same soil wetness, same satellite angle. If we randomly put some chips of an event in training and others in test, the method can score well just by recognising that event's "look". That is cheating without meaning to, and it is called **leakage**.

**Our fix.** We split by whole flood events. Everything from one event goes to one pile.

| Pile | Events |
|---|---|
| **Test** | India (our target region) and Bolivia |
| **Validation** | Sri-Lanka and Ghana |
| **Train** | the other seven events |

So when we report a score on test, the method has never seen anything from those floods. That is why we say "events not used for fitting". A method that does well here has really generalised.

## 5. Units: what we actually answer for

We have no real village boundaries, so we make two kinds of stand-in:

* **Grid units**: the study block cut into **36** equal squares.
* **Chip units** (pseudo-villages): each chip cut into 4 pieces, giving **528** units (336 train, 96 validation, 96 test).

They are not real villages. They give us something village-sized to practise the whole pipeline on.

## 6. Roads and travel time

Layer 4 needs to know how long it takes to drive between places, so Layer 1 builds a **road graph** from OpenStreetMap: junctions are points (**nodes**), road pieces between them are lines (**edges**), each with a travel time. Our study block has **2,096 nodes and 2,567 edges**. A standard method called **Dijkstra's algorithm** finds the fastest route from one place to every other.

Note it minimises **minutes, not kilometres**. A longer road that is faster can win. Layer 4 later makes roads slower where they are probably flooded.

## 7. Receipts: proving nothing was changed

Every file Layer 1 writes gets a **fingerprint**, a short code computed from the file's contents (SHA-256). Change one number in the file and the fingerprint changes completely. Fingerprints are stored in a small file called a **receipt**. Later layers and the tool can check the receipt and so show that the data they display is exactly what was produced.

## 8. Fair sampling: giving each place its true weight

Suppose officers only check fields near roads, because those are easy to reach. Then the checks are not a fair picture of all fields, and a plain average will be wrong.

The fix is an old, simple idea. For every place that could be checked, we record the **inclusion probability**: the known chance that it was picked. A place that was unlikely to be picked but was picked stands in for many similar places, so it counts for more. A place that was very likely to be picked counts for less. Weighting each check by **1 divided by its inclusion probability** (the **Horvitz-Thompson** method) removes the bias.

Small example: if easy places are picked far more often, a plain average of checks over-represents easy places. The weighted average comes back to the true answer.

Layer 1 sets up this machinery, and Layers 3 and 4 use it.

## 9. Revisit gaps: how long was it flooded?

The satellite does not take a picture every day. It passes every few days. If a field is flooded in one picture and dry in the next, all we know is that the flood ended **somewhere in between**, not exactly when. So flood duration is known only as a **range**, called a **censored interval**. Layer 1 measures how long the gaps between passes really are, and Layer 2 uses that to reason about duration.

## 10. Key numbers

| Thing | Value |
|---|---|
| Loss line **tau** | **0.25** (25 percent of paddy area flooded) |
| Sentinel-1 | radar, sees through cloud; water is dark, below about **-15 dB** |
| Dataset | Sen1Floods11, **11** flood events, chips of 128 x 128 pixels at 20 m |
| Chips used after filtering | **121** |
| Split by event | test: India, Bolivia; validation: Sri-Lanka, Ghana; train: the other 7 |
| Units | 36 grid units; 528 chip units (336 / 96 / 96) |
| Road graph | **2,096** nodes, **2,567** edges |
| Fingerprint | SHA-256 |
| Random seed | 20261118 |

## 11. Glossary

* **Chip**: a small radar picture with hand-marked water.
* **dB (decibel)**: log scale for radar brightness. Lower is darker.
* **Event**: one real flood,.
* **Inclusion probability**: known chance a place is selected for checking.
* **Leakage**: test data secretly resembling training data, making scores look better than they are.
* **Loss line (tau)**: the 25 percent threshold.
* **Polarisation (VV, VH)**: direction of the radar signal sent and received.
* **Receipt**: a small file of fingerprints proving data was not altered.
* **Unit**: the thing we give a call for (village or stand-in).

## 12. Review questions

1. **What does Layer 1 do?** It gathers and prepares the data, builds the road graph and units, splits the data fairly and records fingerprints. It predicts nothing.
2. **Why radar and not normal photos?** Radar works through cloud and at night, which matters in the monsoon.
3. **What are VV and VH?** Two readings from the same radar: sent vertical and received vertical or horizontal. VV is best for open water.
4. **Why does water look dark?** Calm water reflects the signal away from the satellite.
5. **Why split by event, not by tile?** Chips from one event look alike. Mixing them across piles lets a method score well by recognising the event.
6. **What are the test events?** India and Bolivia.
7. **Why record inclusion probabilities?** So checks taken in convenient places can be weighted back to a fair answer.
8. **Why do we only know duration as a range?** The satellite visits every few days, so we only know the flood ended between two passes.
9. **What does a receipt do?** It lets anyone check that a file is exactly what Layer 1 produced.